# Smart Money Signal — Validation Notebook

**Components:** Bulk score (60%) + Delivery score (40%), min-max normalized within tier.  
**v1 source:** `~/alpha-signal/data/smart_money/smart_money_score.csv`  
**Key validated signal:** Delivery % (t=2.49 SMALL cap)

In [ ]:
import sys, os
os.chdir(os.path.expanduser("~/alpha-signal-v2"))
sys.path.insert(0, ".")

import pandas as pd
from signals.smart_money import _load_data, _compute_scores

stocks, bulk, delivery = _load_data()
v2 = _compute_scores(stocks, bulk, delivery)
print(f"v2: {len(v2)} stocks, bulk deals in {v2['net_buy_qty'].notna().sum()} stocks")
print(f"Score: mean={v2['smart_money_score'].mean():.1f}, range=[{v2['smart_money_score'].min():.1f}, {v2['smart_money_score'].max():.1f}]")

v1 = pd.read_csv(os.path.expanduser("~/alpha-signal/data/smart_money/smart_money_score.csv"))
merged = v2.merge(v1[["sid", "smart_money_score", "bulk_score", "delivery_score"]],
                  on="sid", how="inner", suffixes=("_v2", "_v1"))

for label, v2c, v1c in [("Delivery", "delivery_score_v2", "delivery_score_v1"),
                         ("Composite", "smart_money_score_v2", "smart_money_score_v1")]:
    both = merged[[v2c, v1c]].dropna()
    corr = both.iloc[:, 0].corr(both.iloc[:, 1])
    diff = (both.iloc[:, 0] - both.iloc[:, 1]).abs().mean()
    print(f"{label}: {len(both)} stocks, corr={corr:.4f}, mean|diff|={diff:.1f}")

## Save to DB (only if validation passed)

In [ ]:
# from signals.smart_money import compute
# compute(dry_run=False)